# Demonstration Script: Melt Pool Contour Prediction from Surface Thermal Images

In [ ]:
import json
import os
import torch
import numpy as np
import wandb
from matplotlib import pyplot as plt
import glob
from torch.utils.data import DataLoader
from experimental_depth_prediction.utils.utils import get_dataset


### Configuring file paths

In [ ]:
current_folder = os.path.dirname(os.path.abspath(''))
root_folder = current_folder
dataset_folder = os.path.join(root_folder, 'data/sequence_exposure')
path_to_sample = os.path.join(dataset_folder, 'sample')

### Loading sample data

In [ ]:
## Load the frames of one process condition from the sample folder
print('Process conditions in sample folder:', sorted(os.listdir(path_to_sample)))

sample_pv = '400_0.6'  # pick any folder listed above
p, v = sample_pv.split('_')

print(f'Using P = {p} W, V = {v} m/s')

def frame_number(file_path):
    """Frame files are named by frame number, e.g. 12.npy."""
    return int(os.path.splitext(os.path.basename(file_path))[0])

data_file_paths = {}
for image_type in ['Temp', 'Mono']:
    for exposure in ['4', '20']:
        files = glob.glob(os.path.join(path_to_sample, sample_pv, image_type, exposure, '*.npy'))
        data_file_paths[f'{image_type.lower()}_{exposure}'] = sorted(files, key = frame_number)
        print(f'{image_type} frames at {exposure} µs: {len(files)}')


##  Melt Pool Surface Images

In [ ]:
# Use one sample from the data file paths dictionary

exposure = 20 # options: 4 or 20
image_type = 'mono' # options: temp for temperature, mono for monochrome
start = 0 # adjusts cropping of melt pool within frame
frame_index = 20 # index of the sequence to plot 
colorbar = True

plt.figure(dpi=300)

if image_type == 'temp':
    temperature_values = np.load(data_file_paths[f'{image_type}_{exposure}'][frame_index]).astype('float')/10 # divide by 10 to convert to K
    cropped_meltpool = temperature_values[:-1, 50:150]
    cropped_meltpool[cropped_meltpool == 0] = np.nan
    plt.title(f'Temperature, Exposure: {exposure} µs, Index: {frame_index}')
    plt.imshow(cropped_meltpool, cmap='inferno', alpha=1, extent=[start, start + 100, 0, cropped_meltpool.shape[0]], vmin = 2000, vmax=3500)
    plt.axis('off')
    if colorbar:
        plt.colorbar()
    plt.show()

elif image_type == 'mono':
    plt.title(f'Monochrome, Exposure: {exposure} µs, Index: {frame_index}')
    monochrome_values = np.load(data_file_paths[f'{image_type}_{exposure}'][frame_index]).astype('float')
    cropped_meltpool = monochrome_values[:-1, 50:150]
    cropped_meltpool[cropped_meltpool <  20000] = np.nan
    plt.title(f'Monochrome, Exposure: {exposure} µs, Index: {frame_index}')

    plt.imshow(cropped_meltpool, cmap='inferno', alpha=1, extent=[start, start + 100, 0, cropped_meltpool.shape[0]], vmin = 20000, vmax=65000)
    if colorbar:
        plt.colorbar()
   
    plt.axis('off')
    plt.show()

##  Melt Pool Depth Contours

In [ ]:
from experimental_depth_prediction.utils.plotting_utils import frame_tick
from matplotlib import pyplot as plt


depth_dataset = os.path.join(root_folder, 'data', 'input_label_map')
with open(os.path.abspath(os.path.join(depth_dataset, "64_enlarged_sdf_contour_area.json")), "r", encoding="utf-8") as fp:
    sample_depth = json.load(fp)    
power, velocity = p, v
p_v = sample_pv
resolution = 64 # fixed, based on SDF calculation


# melt pool coordinates are rescaled from [0, 1] to the actual dimensions in µm
# mesh_x_conversion, mesh_y_conversion correspond to the largest width and depths observed
MESH_X_CONVERSION = 368.208
MESH_Y_CONVERSION = 450.65 
p_v = f'{power}_{velocity}'

x = np.arange(resolution)/resolution
y = np.arange(resolution)/resolution
xx,yy = np.meshgrid(x,y)

# Plot melt pool contour 
plt.figure(dpi = 150, figsize = np.array([4,3])*1.15)
plt.contour(MESH_X_CONVERSION*xx,
            -MESH_Y_CONVERSION*yy,
            np.array(sample_depth[p_v]).reshape(resolution,resolution), 
            origin = 'lower',  
            levels = [0],
              cmap = 'inferno')


plt.xlabel(r'X [$\mu m$]')
plt.ylabel(r'Z [$\mu m$]')
plt.axis('equal')

plt.ylim([-460,0])
plt.gca().set_aspect('equal', adjustable='box')
frame_tick()

plt.title(f'Power = {power} W, V = {velocity} m/s')
plt.show()

### ML Model Demonstration

*Step 1: Loading dataloader and initializing data.*

In this step, a `config` file is used to initialize an instance of a custom PyTorch DataLoader. A sample `config` file is provided below for convenience. Each training run saves its config file as config.yml in its folder under `runs/`. Alternatively, the `config` file can be manually edited to reflect the details of the model training experiment. 


In [ ]:
# Automatically load an experiment from WandB
import yaml

use_wandb = False
if use_wandb: 
    username =  None # insert your specific WandB username here
    run_id =  None # 8 character run ID auto-generated by WandB
    project_name =  None # project name 
    api = wandb.Api()
    run_path = f"{username}/{project_name}/{run_id}"
    run = api.run(run_path)
    config = run.config # Config information stored in this dictionary
else:
    # Load directly from .yml file
    sample_configuration_file = os.path.join(root_folder, 
                                             'experimental_depth_prediction/config/sample/sample_configuration_temporal_transformer.yml')
        
    with open(sample_configuration_file, 'r') as file:
        config = yaml.safe_load(file)

In [ ]:

sample_dataset = get_dataset(config, 'sample', root_folder = root_folder)
test_dataloader = DataLoader(sample_dataset, batch_size = config['batch_size'], shuffle = True)
sample_batch = next(iter(test_dataloader))


#### Step 2: Loading in trained ML Model

In this step, a `config` file is used to initialize an instance of the temporal transformer model. The sample `config` file provided earlier is reused to specify the model configuration. The pretrained model is provided as a PyTorch model checkpoint.

In [ ]:
from experimental_depth_prediction.utils.utils import load_model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config['data_parallel'] = False
model_artifact_path = os.path.join(root_folder, 'checkpoints', 'sample_temporal_transformer.pth')
model = load_model(model_type = config['model_type'],
                       run_path =  None,
                       config = config,
                       model_artifact_path=model_artifact_path,
                       device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
                       train = False)
model.to(device)

#### Step 3: Sample predicted melt pool cross-sections

In [ ]:
import numpy as np
import seaborn as sns
from skimage import measure
from experimental_depth_prediction.utils.utils import sdf_to_binary
total_number = 0
with torch.no_grad():
    for sample_batch in test_dataloader:

        sample_output = model(sample_batch[0].to(device))
    
        binary_pred = sdf_to_binary(sample_output[0].reshape(64,64).cpu().detach().numpy(), 1024)
        binary_target = sdf_to_binary(sample_batch[-2][0].reshape(64,64).cpu().detach().numpy(), 1024)
        ground_truth = binary_target
        predicted =  binary_pred
            
        contour_gt = measure.find_contours(ground_truth)
        contour_pred = measure.find_contours(predicted)
        sns.set(style="ticks", color_codes=True)
        plt.figure(dpi = 150, figsize = np.array([4,3])*1.2)
        plt.fill(contour_pred[0][:, 1]/1024, -contour_pred[0][:, 0]/1024, linewidth=2, alpha = 0.2)
        plt.fill(contour_gt[0][:, 1]/1024,  -contour_gt[0][:, 0]/1024,  linewidth=2, alpha = 0.2)
        plt.plot(contour_pred[0][:, 1]/1024,-contour_pred[0][:, 0]/1024, linewidth=2, alpha = 1, label = 'Predicted')
        plt.plot(contour_gt[0][:, 1]/1024,  -contour_gt[0][:, 0]/1024,  linewidth=2, alpha = 1, label = 'Ground Truth')

        frame_tick()

        plt.ylim([-1, 0])
        plt.xlim([0, 1])
        ax = plt.gca()

        ax.set_aspect(abs((ax.get_ylim()[1] - ax.get_ylim()[0]) / (ax.get_xlim()[1] - ax.get_xlim()[0])))
        plt.ylabel(r'Normalized Depth')
        plt.xlabel(r'Normalized Width')
        plt.legend(frameon = False, loc = 'lower right')
        plt.show()
        total_number += 1
        if total_number > 4:
            break
